# Sarvam-1 SFT on Odia GSM8K

Supervised fine-tuning of `sarvamai/sarvam-1` on the `train` split of `tripathysagar/odia-gsm8k`.

- TRL `SFTTrainer` with **QLoRA** (4-bit) for single-GPU training on RunPod.
- Training metrics → **Comet ML** (set `COMET_API_KEY` to enable).
- Sanity-check rollouts traced to **Opik** (optional, set `OPIK_API_KEY`).
- Final adapter merged into the base model and pushed to the HF Hub.

## 1. Setup

In [1]:
# %pip install -q --upgrade transformers trl peft bitsandbytes accelerate datasets python-dotenv comet_ml opik huggingface_hub

In [2]:
import gc
import os
import time
from pathlib import Path

# Put HF cache on the RunPod persistent volume so model weights survive pod restart
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")

# Import comet_ml *before* torch so Comet can auto-instrument the framework.
# (Comet warns and disables some autologging if torch is imported first.)
import comet_ml

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from huggingface_hub import login as hf_login, create_repo

print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

PyTorch: 2.8.0+cu128
CUDA   : True NVIDIA L40S


## 2. Configuration

In [3]:
import sys
from pathlib import Path

# Make the repo root importable so `config.py` resolves whether this notebook's
# working dir is the repo root or the notebooks/ subfolder.
_root = Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import *   # all non-secret params (config.py also loads .env secrets)

# Local aliases — keep the names the cells below already use
BASE_MODEL_ID    = MODEL_ID
COMET_PROJECT    = COMET_PROJECT_NAME
COMET_TAGS_EXTRA = COMET_TAGS

# Fail fast on missing push targets — don't waste hours of training
if PUSH_TO_HUB:
    assert SFT_HUB_MODEL_ID, (
        "PUSH_TO_HUB=True but neither SFT_HUB_MODEL_ID nor HF_USERNAME is set. "
        "Set one in config.py before training, or set PUSH_TO_HUB=False."
    )
    assert SFT_ADAPTER_HUB_MODEL_ID, (
        "PUSH_TO_HUB=True but SFT_ADAPTER_HUB_MODEL_ID could not be resolved. "
        "Set HF_USERNAME or SFT_ADAPTER_HUB_MODEL_ID in config.py."
    )

SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base model : {BASE_MODEL_ID}")
print(f"Dataset    : {DATASET_ID} ({TRAIN_SPLIT})")
print(f"Output dir : {SFT_OUTPUT_DIR}")
print(f"Hub repo   : {SFT_HUB_MODEL_ID or '(unset)'}  push={PUSH_TO_HUB}")
print(f"Adapter    : {SFT_ADAPTER_HUB_MODEL_ID or '(unset)'}")
print(f"QLoRA      : {USE_QLORA}  GPU: {GPU_TYPE or '(not set)'}")
print(f"Epochs={NUM_EPOCHS}  lr={LEARNING_RATE}  bs={BATCH_SIZE}x{GRAD_ACCUM}  max_seq={MAX_SEQ_LEN}")

Base model : sarvamai/sarvam-1
Dataset    : tripathysagar/odia-gsm8k (train)
Output dir : /workspace/sarvam1-odia-gsm8k-sft
Hub repo   : pareshppp/sarvam-1-odia-gsm8k-sft  push=True
Adapter    : pareshppp/sarvam-1-odia-gsm8k-sft-adapter
QLoRA      : True  GPU: l40s
Epochs=2  lr=0.0003  bs=16x1  max_seq=2048


## 3. Authenticate Hub + Tracking

In [4]:
if HF_TOKEN:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF Hub : logged in")
else:
    print("HF Hub : HF_TOKEN not set — push will fail")

if COMET_API_KEY:
    tags = ["sft", "qlora" if USE_QLORA else "bf16"]
    if GPU_TYPE:
        tags.append(GPU_TYPE)
    if COMET_TAGS_EXTRA:
        tags.extend(t.strip() for t in COMET_TAGS_EXTRA.split(",") if t.strip())
    os.environ["COMET_PROJECT_NAME"] = COMET_PROJECT
    os.environ["COMET_TAGS"]         = ",".join(tags)
    if COMET_WORKSPACE:
        os.environ["COMET_WORKSPACE"] = COMET_WORKSPACE
    comet_ml.login(api_key=COMET_API_KEY)   # auth only — project/workspace/tags set via env vars above
    REPORT_TO = "comet_ml"
    print(f"Comet  : configured  project={COMET_PROJECT}  tags={tags}")
else:
    REPORT_TO = "none"
    print("Comet  : not configured (set COMET_API_KEY to enable)")

# Opik for LLM-trace logging (used in sanity-check rollouts below)
OPIK_ENABLED = bool(OPIK_API_KEY)
if OPIK_ENABLED:
    import opik
    os.environ["OPIK_PROJECT_NAME"] = OPIK_PROJECT_NAME   # Opik reads this automatically
    opik.configure(api_key=OPIK_API_KEY, workspace=OPIK_WORKSPACE)
    print(f"Opik   : configured  project={OPIK_PROJECT_NAME}")
else:
    print("Opik   : not configured (optional)")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF Hub : logged in


COMET INFO: Valid Comet API Key saved in /root/.comet.config (set COMET_CONFIG to change where it is saved).


Comet  : configured  project=odia-finetuning-inference  tags=['sft', 'qlora', 'l40s']
Opik   : not configured (optional)


## 4. Load & Format Dataset

In [5]:
ds = load_dataset(DATASET_ID, split=TRAIN_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(ds)} training examples")
print("Columns:", ds.column_names)
print("\nSample:")
print(ds[0])

Loaded 7473 training examples
Columns: ['id', 'question', 'answer', 'odia_question', 'odia_answer']

Sample:
{'id': 0, 'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'odia_question': 'ନଟାଲିଆ ଏପ୍ରିଲରେ ନିଜର 48 ଜଣ ବନ୍ଧୁଙ୍କୁ କ୍ଲିପ ବିକ୍ରି କଲେ, ଏବଂ ମଇରେ ତାହାଠାରୁ ଅଧା ସଂଖ୍ୟା କ୍ଲିପ ବିକ୍ରି କରିଥିଲେ। ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ ନଟାଲିଆ ମୋଟ କେତେ କ୍ଲିପ ବିକ୍ରି କରିଥିଲେ?', 'odia_answer': 'ନଟାଲିଆ ମଇରେ 48/2 = <<48/2=24>>24 କ୍ଲିପ ବିକ୍ରି କଲେ।\nନଟାଲିଆ ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ 48+24 = <<48+24=72>>72 କ୍ଲିପ ବିକ୍ରି କଲେ।\n#### 72'}


In [6]:
SYSTEM_PROMPT = (
    "ଆପଣ ଜଣେ ସହାୟକ ଗଣିତ ସହକାରୀ ଅଟନ୍ତି। "
    "ତଳେ ଦିଆଯାଇଥିବା ସମସ୍ୟାକୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ସମାଧାନ କରନ୍ତୁ। "
    "ଶେଷରେ, ଆପଣଙ୍କର ଚୂଡ଼ାନ୍ତ ସାଂଖ୍ୟିକ ଉତ୍ତରକୁ ଏକ ନୂଆ ଧାଡ଼ିରେ '####' ସହିତ ଆରମ୍ଭ କରି ଲେଖନ୍ତୁ।"
)
print("SYSTEM_PROMPT defined. Dataset will be formatted after tokenizer load.")

SYSTEM_PROMPT defined. Dataset will be formatted after tokenizer load.


## 5. Load Model + Tokenizer

In [7]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")
model_kwargs = dict(
    token=HF_TOKEN,
    trust_remote_code=True,
    device_map="auto",
    attn_implementation="flash_attention_2",  # required for correct doc isolation when packing
)

if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["torch_dtype"] = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
model.config.use_cache = False  # incompatible with gradient checkpointing

if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("Model ready.")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Model ready.


In [8]:
EOS = tokenizer.eos_token

def format_example(example):
    q = example[QUESTION_COL]
    a = example[ANSWER_COL]
    return {"text": f"{SYSTEM_PROMPT}\n\nପ୍ରଶ୍ନ: {q}\nଉତ୍ତର: {a}{EOS}"}

train_ds = ds.map(format_example, remove_columns=ds.column_names)
print(f"Formatted {len(train_ds)} examples.")
print("\nFirst formatted sample (truncated):")
print(train_ds[0]["text"][:600], "...")

Formatted 7473 examples.

First formatted sample (truncated):
ଆପଣ ଜଣେ ସହାୟକ ଗଣିତ ସହକାରୀ ଅଟନ୍ତି। ତଳେ ଦିଆଯାଇଥିବା ସମସ୍ୟାକୁ ପର୍ଯ୍ୟାୟକ୍ରମେ ସମାଧାନ କରନ୍ତୁ। ଶେଷରେ, ଆପଣଙ୍କର ଚୂଡ଼ାନ୍ତ ସାଂଖ୍ୟିକ ଉତ୍ତରକୁ ଏକ ନୂଆ ଧାଡ଼ିରେ '####' ସହିତ ଆରମ୍ଭ କରି ଲେଖନ୍ତୁ।

ପ୍ରଶ୍ନ: ନଟାଲିଆ ଏପ୍ରିଲରେ ନିଜର 48 ଜଣ ବନ୍ଧୁଙ୍କୁ କ୍ଲିପ ବିକ୍ରି କଲେ, ଏବଂ ମଇରେ ତାହାଠାରୁ ଅଧା ସଂଖ୍ୟା କ୍ଲିପ ବିକ୍ରି କରିଥିଲେ। ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ ନଟାଲିଆ ମୋଟ କେତେ କ୍ଲିପ ବିକ୍ରି କରିଥିଲେ?
ଉତ୍ତର: ନଟାଲିଆ ମଇରେ 48/2 = <<48/2=24>>24 କ୍ଲିପ ବିକ୍ରି କଲେ।
ନଟାଲିଆ ଏପ୍ରିଲ ଓ ମଇ ମିଳାଇ 48+24 = <<48+24=72>>72 କ୍ଲିପ ବିକ୍ରି କଲେ।
#### 72</s> ...


## 6. Configure Trainer

In [9]:
run_name = "-".join(filter(None, ["sarvam1-sft", GPU_TYPE, str(int(time.time()))]))

sft_config = SFTConfig(
    output_dir=str(SFT_OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,   # float in (0,1) is interpreted as a ratio of total steps
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    max_length=MAX_SEQ_LEN,
    packing=USE_PACKING,
    dataset_text_field="text",
    report_to=REPORT_TO,
    run_name=run_name,
    push_to_hub=False,    # we merge + push manually below
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"Train examples   : {len(train_ds)}")
print(f"Trainable params : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"Run name         : {run_name}")

Tokenizing train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/7473 [00:00<?, ? examples/s]

Train examples   : 7473
Trainable params : 23,969,792 / 2,549,057,536 (0.94%)
Run name         : sarvam1-sft-l40s-1782447069


## 7. Train

In [10]:
trainer.train()
print("Training complete.")

adapter_dir = SFT_OUTPUT_DIR / "final-adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter saved to {adapter_dir}")

if PUSH_TO_HUB:
    print(f"Pushing adapter → {SFT_ADAPTER_HUB_MODEL_ID}")
    create_repo(SFT_ADAPTER_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO, exist_ok=True)
    trainer.model.push_to_hub(SFT_ADAPTER_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    tokenizer.push_to_hub(SFT_ADAPTER_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    print(f"Pushed adapter: https://huggingface.co/{SFT_ADAPTER_HUB_MODEL_ID}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/paresh-pradhan/odia-finetuning-inference/f19b5a5069d6401ea53a34b5f045680f

[transformers] Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss
10,1.547893
20,0.956703
30,0.844028
40,0.786175
50,0.765527
60,0.749596
70,0.729669
80,0.705504
90,0.704840
100,0.698502


Training complete.
Adapter saved to /workspace/sarvam1-odia-gsm8k-sft/final-adapter
Pushing adapter → pareshppp/sarvam-1-odia-gsm8k-sft-adapter


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed adapter: https://huggingface.co/pareshppp/sarvam-1-odia-gsm8k-sft-adapter


## 8. Merge LoRA → Push to HF Hub

In [11]:
merged_model = None

if PUSH_TO_HUB:
    # Free the quantized training model before reloading in bf16 — gc.collect() is required
    # because del alone doesn't release LoRA / accelerator wrapper references quickly enough
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Reloading base model in bf16 to merge adapter...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    merged_model = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()

    print(f"Pushing merged model → {SFT_HUB_MODEL_ID}")
    create_repo(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO, exist_ok=True)
    merged_model.push_to_hub(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    tokenizer.push_to_hub(SFT_HUB_MODEL_ID, token=HF_TOKEN, private=PRIVATE_REPO)
    print(f"Pushed: https://huggingface.co/{SFT_HUB_MODEL_ID}")
else:
    print("PUSH_TO_HUB=false — adapter saved locally only.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reloading base model in bf16 to merge adapter...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Pushing merged model → pareshppp/sarvam-1-odia-gsm8k-sft


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed: https://huggingface.co/pareshppp/sarvam-1-odia-gsm8k-sft


## 9. Sanity-Check Inference

In [12]:
test_ds = load_dataset(DATASET_ID, split="test", token=HF_TOKEN).select(range(3))

eval_model = merged_model if merged_model is not None else trainer.model
eval_model.eval()

_track = opik.track(name="sft_sanity_check") if OPIK_ENABLED else (lambda f: f)

@_track
def run_sample(question: str) -> str:
    prompt = f"{SYSTEM_PROMPT}\n\nପ୍ରଶ୍ନ: {question}\nଉତ୍ତର:"
    inputs = tokenizer(prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

for ex in test_ds:
    pred = run_sample(ex[QUESTION_COL])
    print("Q   :", ex[QUESTION_COL][:120], "...")
    print("Gold:", str(ex[ANSWER_COL])[:200], "...")
    print("Pred:", pred[:300])
    print("-" * 60)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q   : ଜାନେଟଙ୍କ ବତକ ପ୍ରତିଦିନ 16 ଟି ଡିମ୍ବ ଦଦଦଦଦିଅନ୍ତି। ସେ ପ୍ରତି ସକାଳେ ନାସ୍ତା ପାଇଁ 3 ଟି ଖାନ୍ତି ଏବଂ ପ୍ରତିଦିନ 4 ଟି ଦିଆଁ ସାହିମାନଙ୍କ  ...
Gold: ଜାନେଟ ପ୍ରତିଦିନ 16 - 3 - 4 = <<16-3-4=9>>9 ଟି ବତକ ଡିମ୍ବ ବିକ୍ରି କରନ୍ତି।
ସେ କୃଷକ ବଜାରରେ ପ୍ରତିଦିନ 9 * 2 = $<<9*2=18>>18 ରୋଜଗାର କରନ୍ତି।
#### 18 ...
Pred: ଜାନେଟ୍‌ଙ୍କ ପାଖରେ 16 + 4 = <<16+4=20>>20 ଟି ଡିମ୍ବ ଅଛି।
ସେ ସେଗୁଡ଼ିକୁ ପ୍ରତିଦିନ 20 / 2 = <<20/2=10>>10 ଟି ଦିଏ।
ତେଣୁ, ସେ ପ୍ରତ୍ୟେକ ଦିନ 10 - 3 = <<10-3=7>>7 ଟି ମଫିନ ବେକ କରନ୍ତି।
ତେଣୁ, ସେ ପ୍ରତ୍ୟେକ ଦିନ 7 * 4 = <<7*4=28>>28 ଡଲାର ରୋଜଗାର କରନ୍ତି।
ତେଣୁ, ସେ କୃଷକ ବଜାରରେ ପ୍ରତିଦିନ 28 * 2 = <<28*2=56>>56 ଡଲାର ରୋଜଗାର କର
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q   : ଏକ ବସ୍ତ୍ର ନିର୍ମାଣରେ 2 ବୋଲ୍ଟ ନୀଳ ଫାଇବର ଏବଂ ତାହାର ଅଧା ପରିମାଣ ଧଳା ଫାଇବର ଲାଗେ। ମୋଟ କେତେ ବୋଲ୍ଟ ଲାଗେ? ...
Gold: ଏହାକୁ 2/2=<<2/2=1>>1 ବୋଲ୍ଟ ଧଳା ଫାଇବର ଲାଗେ।
ତେଣୁ ମୋଟ କପଡ଼ା 2+1=<<2+1=3>>3 ବୋଲ୍ଟ।
#### 3 ...
Pred: ଧଳା ଫାଇବର = 1/2 * 2 = <<1/2*2=1>>1 ବୋଲ୍ଟ
ନୀଳ ଫାଇବର = 2 + 1 = <<2+1=3>>3 ବୋଲ୍ଟ
ମୋଟ = 3 + 1 = <<3+1=4>>4 ବୋଲ୍ଟ
#### 4
------------------------------------------------------------
Q   : ଜଶ ଏକ ଘର ଫ୍ଲିପ କରିବାକୁ ଚେଷ୍ଟା କରନ୍ତି। ସେ $80,000 ରେ ଘର କିଣି $50,000 ମରାମତି କରନ୍ତି। ଏହା ଘରର ମୂଲ୍ୟ 150% ବୃଦ୍ଧି କଲା। ସେ କେତ ...
Gold: ଘର ଏବଂ ମରାମତିର ମୋଟ ଖର୍ଚ୍ଚ 80,000+50,000=$<<80000+50000=130000>>130,000 ହେଲା।
ସେ ଘରର ମୂଲ୍ୟ 80,000*1.5=<<80000*1.5=120000>>120,000 ବୃଦ୍ଧି କଲେ।
ତେଣୁ ଘରର ନୂଆ ମୂଲ୍ୟ 120,000+80,000=$<<120000+80000=200000>>2 ...
Pred: ଘରର ମୂଲ୍ୟ 150% ବୃଦ୍ଧି ହେତୁ, ନୂତନ ମୂଲ୍ୟ $80,000 + ($80,000 * 1.5) = $<<80000*1.5=120000>>120,000।
ସେ $120,000 - $80,000 = $<<120000-80000=40000>>40,000 ଲାଭ କଲେ।
#### 40000
------------------------------------------------------------


## 10. Finalize Tracking

In a Jupyter environment Comet does not auto-close the run, so we end it
explicitly to flush all metrics, parameters and code before exiting.

In [13]:
# Close the Comet experiment so all data is logged before the kernel exits.
if REPORT_TO == "comet_ml":
    comet_ml.end()
    print("Comet  : experiment ended — all metrics & code flushed")

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : sarvam1-sft-l40s-1782447069
COMET INFO:     url                   : https://www.comet.com/paresh-pradhan/odia-finetuning-inference/f19b5a5069d6401ea53a34b5f045680f
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss [13]                      : (0.6897125244140625, 1.7364885807037354)
COMET INFO:     train/entropy [14]             : (0.6920365810394287, 1.4320055723190308)
COMET INFO:     train/epoch [14]               : (0.14705882352941177, 2.0)
COMET INFO:     train/grad_norm [13]           : (0.09033203125, 0.423828125)
COMET INFO:     train/learning_rate [13]       : (2.174330943914593e-06, 0.0002998221082861234)
COMET INFO:     train/

Comet  : experiment ended — all metrics & code flushed
